# Notebook 01 — Tracking Pipeline

This notebook walks through every stage of the tracking pipeline interactively.
It is equivalent to running `scripts/run_tracking.py` but lets you inspect
intermediate outputs at each step.

## Stages
1. Setup & configuration
2. (Optional) Background subtraction
3. Draw vial ROIs
4. RF-DETR + OC-SORT tracking → wide CSV
5. Hungarian stitching → stitched long CSV
6. Vial assignment + compact IDs → compact_tracks.csv
7. Overlay video rendering

**Replace all `PLACEHOLDER` paths with your actual file paths.**

In [2]:
import sys
sys.path.insert(0, '../src')

import json
import os
import cv2
import yaml
import pandas as pd
from pathlib import Path
from IPython.display import Video

from preprocessing import preprocess_bgsub_gui
#from src.tracking import export_tracks_xy_tuple_csv_one_config
#from src.stitching import wide_to_long, build_tracklets, stitch_per_vial
#from src.roi import draw_and_save_vial_rois, assign_vials_and_compact_ids
#from src.visualization import render_vial_overlay_video

## 1 — Configuration

Set your paths and Roboflow credentials here.

In [ ]:
# ---- EDIT THESE ----
RAW_VIDEO = r"../2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m/13 DPE/001/2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted.mp4"
API_KEY   = ""
MODEL_ID  = "flies-123/1"   # e.g. "flies-123/1"

# Load defaults from config.yaml (override below if needed)
with open("../config.yaml") as _f:
    _cfg = yaml.safe_load(_f)
_t = _cfg.get("tracker", {})
_s = _cfg.get("stitching", {})
_p = _cfg.get("preprocessing", {})

confidence              = _t.get("confidence", 0.10)
lost_track_buffer       = _t.get("lost_track_buffer", 90)
min_matching_threshold  = _t.get("minimum_matching_threshold", 0.01)
min_consecutive_frames  = _t.get("minimum_consecutive_frames", 10)
asso_func               = _t.get("asso_func", "hmiou")
n_flies_per_vial        = _s.get("n_flies_per_vial", 15)
max_gap_frames          = _s.get("max_gap_frames", 90)
bg_gain                 = _p.get("bg_gain", 1.2)
bg_white_level          = _p.get("bg_white_level", 245)
bg_percentile           = _p.get("bg_percentile", 85.0)
bg_sample_stride        = _p.get("bg_sample_stride", 1)
default_end             = _p.get("default_end", 700)

# Auto-increment output directory: run_1, run_2, run_3, ...
_outputs_root = Path("../outputs")
_outputs_root.mkdir(parents=True, exist_ok=True)
_existing = [d for d in _outputs_root.iterdir() if d.is_dir() and d.name.startswith("run_")]
_next_n = max((int(d.name.split("_")[1]) for d in _existing if d.name.split("_")[1].isdigit()), default=0) + 1
OUTPUT_PATH = str(_outputs_root / f"run_{_next_n}")

os.makedirs(OUTPUT_PATH, exist_ok=True)
PATH_TO_VID = RAW_VIDEO
print("Output dir:", OUTPUT_PATH)
print(f"asso_func={asso_func}, n_flies_per_vial={n_flies_per_vial}, max_gap_frames={max_gap_frames}")
print(f"bg_gain={bg_gain}, bg_white_level={bg_white_level}, bg_percentile={bg_percentile}, bg_sample_stride={bg_sample_stride}, default_end={default_end}")

## 2 — (Optional) Background subtraction

Opens a GUI: draw a crop ROI and choose a frame range.
The output is a `_pp.mp4` file with the **85th-percentile** background subtracted.
Skip this cell if your video already has good contrast.

In [ ]:
preprocess = True  # set to True to run the GUI
RAW_VIDEO = r"../2024-02-05_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m/13 DPE/001/2024-02-12_NEG-008_hTDP43_WT-A90V-G287S-G294A-A315T-M337V_m_13d_001-converted.mp4"

if preprocess:
    PATH_TO_VID = Path(
        preprocess_bgsub_gui(
            video_path=RAW_VIDEO,
            out_mp4=None,
            default_end=default_end,
            gain=bg_gain,
            white_level=bg_white_level,
            bg_sample_stride=bg_sample_stride,
            bg_percentile=bg_percentile,
        )
    )
    print("Preprocessed video:", PATH_TO_VID)

## 3 — Draw vial ROIs

Opens an OpenCV GUI on frame 0: drag rectangles around each vial.
Press **q** when all 6 ROIs are drawn. Saved to `vial_rois.json`.

This is a one-time step — reuse the JSON for the same experimental setup.

In [ ]:
ROI_JSON = os.path.join(OUTPUT_PATH, "vial_rois.json")

draw_and_save_vial_rois(
    video_path=str(PATH_TO_VID),
    roi_json_path=ROI_JSON,
)

## 4 — RF-DETR + OC-SORT tracking

Runs the detector + tracker on every frame and writes a wide CSV.
This is the most time-consuming step. 

In [ ]:
WIDE_CSV = os.path.join(OUTPUT_PATH, "tracks_wide_format.csv")

df_wide = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=WIDE_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    confidence=confidence,
    lost_track_buffer=lost_track_buffer,
    minimum_matching_threshold=min_matching_threshold,
    minimum_consecutive_frames=min_consecutive_frames,
    asso_func=asso_func,
    max_frames=None,
)

print(df_wide.shape)
df_wide.head()

## 5 — Hungarian stitching

Links fragmented tracklets across gaps using motion-consistent assignment.
Output: long CSV with `orig_id` and `stitched_id` columns.

In [ ]:
STITCHED_CSV = os.path.join(OUTPUT_PATH, "tracks_xy_stitched_long.csv")

# Load vial ROIs
with open(ROI_JSON) as f:
    vial_rois = {k: tuple(v) for k, v in json.load(f).items()}

# Build long df and tracklets
cap = cv2.VideoCapture(str(PATH_TO_VID))
fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
cap.release()

long_df   = wide_to_long(pd.read_csv(WIDE_CSV))
tracklets = build_tracklets(long_df, fps=fps)

print(f"Built {len(tracklets)} tracklets from {long_df['orig_id'].nunique()} original IDs")

# Per-vial iterative stitching
stitched_df = stitch_per_vial(
    long_df              = long_df,
    vial_rois            = vial_rois,
    n_flies_per_vial     = n_flies_per_vial,
    max_gap              = max_gap_frames,
    tracklets            = tracklets,
    min_points_for_scale = min_consecutive_frames,
)

stitched_df.to_csv(STITCHED_CSV, index=False)
print(f"\nSaved: {STITCHED_CSV}")
print(f"Stitched IDs: {stitched_df['stitched_id'].nunique()} (from {stitched_df['orig_id'].nunique()} original)")

## 6 — Vial assignment + compact IDs

Assigns each point to a vial using the ROI JSON, then assigns compact sequential IDs
(left → right within each vial).

In [ ]:
COMPACT_CSV = os.path.join(OUTPUT_PATH, "compact_tracks.csv")

cap = cv2.VideoCapture(str(PATH_TO_VID))
fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
cap.release()

df_compact = assign_vials_and_compact_ids(
    stitched_csv=STITCHED_CSV,
    roi_json=ROI_JSON,
    out_csv=COMPACT_CSV,
    fps=fps,
)

print(df_compact.shape)
df_compact.head()

## 7 — Overlay video

Renders each fly as a coloured dot on the original video.

In [ ]:
OVERLAY_MP4 = os.path.join(OUTPUT_PATH, "overlay_vials_shaded.mp4")

render_vial_overlay_video(
    video_path=str(PATH_TO_VID),
    csv_path=COMPACT_CSV,
    out_mp4=OVERLAY_MP4,
)

Video(OVERLAY_MP4, width=800)